## Extracting halfspaces (hyperplanes) from pairwise choices

Each row is a pairwise choice between **Left** and **Right** options with covariates.
Assuming linear utility \(U(x)=\theta\cdot x\), choosing Left implies:

\[
\theta \cdot (x_L - x_R) > 0
\]

We’ll encode each row as a halfspace constraint in **\(\theta\)-space**:

- If `chosen == 0` (Left chosen): \(a = x_L - x_R\)
- If `chosen == 1` (Right chosen): \(a = x_R - x_L = -(x_L - x_R)\)

So every observation yields \(a \cdot \theta > 0\).



In [10]:
from __future__ import annotations

import csv
from pathlib import Path
from typing import Any

DATA_PATH = Path('tryout.csv')  # assumes notebook is run from the folder containing tryout.csv
CHOICE_COL = 'chosen'          # per your note: choice lives here

# Choose how to encode obesity:
# - 'ordinal' => single scalar feature (theta is 5D total)
# - 'onehot'  => 6-d one-hot block (theta is 10D total)
OBESITY_ENCODING = 'ordinal'

# If obesity is missing in the CSV (coded as -1), we must still output a numeric feature.
# To keep the feature space 5D (no extra "missing" indicator), we impute missing obesity to 0.0.
# If you'd rather drop those rows or use a different imputation, say so.
MISSING_OBESITY_IMPUTE = 0.0

# Numeric covariates (as given)
NUM_FEATURES = [
    ('lifeYearsGained', 'Years of life expected to be gained'),
    ('elderlyDep', 'Number of elderly dependents'),
    ('yearsWaiting', 'Years on waiting list'),
    ('weeklyWorkhours', 'Hours able to work post-transplant'),
]

# Obesity is categorical; in this file it appears coded as integers (0..5), with -1 used in a few rows.
OBESITY_LEVELS = [0, 1, 2, 3, 4, 5]
OBESITY_LABELS = {
    0: 'underweight',
    1: 'normal',
    2: 'overweight',
    3: 'obese',
    4: 'morbidly_obese',
    5: 'very_morbidly_obese',
}


def _to_float(x: Any) -> float:
    if x is None:
        return float('nan')
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return float('nan')
    return float(s)


def _to_int(x: Any) -> int:
    return int(float(str(x).strip()))


def _onehot_int(val: Any, levels: list[int]) -> list[float]:
    """One-hot for an int-coded categorical var. Unknown/-1/blank -> all zeros."""
    try:
        v = _to_int(val)
    except Exception:
        return [0.0] * len(levels)
    if v == -1:
        return [0.0] * len(levels)
    out = [0.0] * len(levels)
    if v in levels:
        out[levels.index(v)] = 1.0
    return out


def x_side(row: dict[str, str], *, side: str, obesity_encoding: str = 'onehot') -> tuple[list[float], list[str]]:
    """Build x_L or x_R for one row; returns (x, feature_names)."""
    if side not in {'l', 'r'}:
        raise ValueError("side must be 'l' or 'r'")

    x: list[float] = []
    names: list[str] = []

    for name, _desc in NUM_FEATURES:
        x.append(_to_float(row[f"{side}_{name}"]))
        names.append(name)

    if obesity_encoding == 'ordinal':
        ob = _to_float(row[f"{side}_obesity"])
        if ob == -1:
            ob = float(MISSING_OBESITY_IMPUTE)
        x.append(ob)
        names.append('obesity')
    elif obesity_encoding == 'onehot':
        x.extend(_onehot_int(row[f"{side}_obesity"], OBESITY_LEVELS))
        names.extend([f"obesity_{OBESITY_LABELS[lvl]}" for lvl in OBESITY_LEVELS])
    else:
        raise ValueError("obesity_encoding must be 'onehot' or 'ordinal'")

    return x, names


def extract_halfspaces_from_csv(
    path: Path,
    *,
    choice_col: str = CHOICE_COL,
    obesity_encoding: str = 'onehot',
) -> tuple[list[list[float]], list[str]]:
    """Return (A, feature_names) where each row a_i defines a halfspace a_i · theta > 0."""
    with path.open('r', newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        raise ValueError('CSV has no rows')

    A: list[list[float]] = []
    feature_names: list[str] | None = None

    for row in rows:
        x_l, names_l = x_side(row, side='l', obesity_encoding=obesity_encoding)
        x_r, names_r = x_side(row, side='r', obesity_encoding=obesity_encoding)
        if names_l != names_r:
            raise RuntimeError('Left/right feature name mismatch')
        if feature_names is None:
            feature_names = names_l

        # delta = x_L - x_R
        delta = [xl - xr for xl, xr in zip(x_l, x_r)]

        # chosen: 0 => left chosen, 1 => right chosen
        c = _to_int(row[choice_col])
        if c not in (0, 1):
            raise ValueError(f"Expected {choice_col} to be 0/1. Got {c} in row {row}")
        sign = 1.0 if c == 0 else -1.0

        # a = sign * (x_L - x_R)
        A.append([sign * d for d in delta])

    assert feature_names is not None
    return A, feature_names


A, feature_names = extract_halfspaces_from_csv(DATA_PATH, obesity_encoding=OBESITY_ENCODING)

print('rows (constraints):', len(A))
print('features (theta dims):', len(feature_names))
print('feature_names:', feature_names)



rows (constraints): 60
features (theta dims): 5
feature_names: ['lifeYearsGained', 'elderlyDep', 'yearsWaiting', 'weeklyWorkhours', 'obesity']


In [11]:
# Pretty-print a few constraints as human-readable inequalities

from typing import Iterable


def format_inequality(a: Iterable[float], names: list[str], *, max_terms: int = 50) -> str:
    terms = []
    for coef, name in zip(a, names):
        if abs(coef) < 1e-12:
            continue
        # show integers cleanly when possible
        if abs(coef - round(coef)) < 1e-12:
            coef_str = str(int(round(coef)))
        else:
            coef_str = f"{coef:.3g}"
        terms.append(f"({coef_str})·θ_{name}")
        if len(terms) >= max_terms:
            terms.append('...')
            break
    lhs = ' + '.join(terms) if terms else '0'
    return lhs + ' > 0'


print('\nFirst 10 halfspace constraints (a_i · theta > 0):')
for i in range(min(10, len(A))):
    print(f"{i:02d}: {format_inequality(A[i], feature_names)}")

# Raw normals for downstream work:
# - A[i] is the normal vector a_i in theta-space
# - The feasible set is {theta : for all i, dot(A[i], theta) > 0}
A




First 10 halfspace constraints (a_i · theta > 0):
00: (-15)·θ_lifeYearsGained + (-1)·θ_elderlyDep + (2)·θ_yearsWaiting + (4)·θ_obesity > 0
01: (-3)·θ_elderlyDep + (-4)·θ_yearsWaiting + (20)·θ_weeklyWorkhours > 0
02: (-2)·θ_yearsWaiting + (10)·θ_weeklyWorkhours + (-3)·θ_obesity > 0
03: (20)·θ_lifeYearsGained + (-2)·θ_elderlyDep + (-4)·θ_yearsWaiting + (20)·θ_weeklyWorkhours + (-1)·θ_obesity > 0
04: (20)·θ_lifeYearsGained + (-3)·θ_elderlyDep + (-4)·θ_yearsWaiting > 0
05: (-5)·θ_lifeYearsGained + (1)·θ_elderlyDep + (4)·θ_yearsWaiting + (-1)·θ_obesity > 0
06: (5)·θ_lifeYearsGained + (3)·θ_elderlyDep + (3)·θ_obesity > 0
07: (-5)·θ_lifeYearsGained + (1)·θ_elderlyDep + (4)·θ_yearsWaiting + (20)·θ_weeklyWorkhours + (-1)·θ_obesity > 0
08: (10)·θ_lifeYearsGained + (-1)·θ_elderlyDep + (-2)·θ_yearsWaiting + (1)·θ_obesity > 0
09: (-5)·θ_lifeYearsGained + (-1)·θ_elderlyDep + (-2)·θ_yearsWaiting + (30)·θ_weeklyWorkhours + (2)·θ_obesity > 0


[[-15.0, -1.0, 2.0, 0.0, 4.0],
 [0.0, -3.0, -4.0, 20.0, 0.0],
 [-0.0, -0.0, -2.0, 10.0, -3.0],
 [20.0, -2.0, -4.0, 20.0, -1.0],
 [20.0, -3.0, -4.0, 0.0, 0.0],
 [-5.0, 1.0, 4.0, -0.0, -1.0],
 [5.0, 3.0, -0.0, -0.0, 3.0],
 [-5.0, 1.0, 4.0, 20.0, -1.0],
 [10.0, -1.0, -2.0, -0.0, 1.0],
 [-5.0, -1.0, -2.0, 30.0, 2.0],
 [15.0, -2.0, -2.0, 30.0, -1.0],
 [-15.0, -1.0, -2.0, 10.0, 1.0],
 [-10.0, 2.0, 2.0, 20.0, 2.0],
 [-15.0, 2.0, -0.0, -20.0, -0.0],
 [-0.0, -0.0, -2.0, 50.0, -0.0],
 [-15.0, 1.0, 4.0, 20.0, 2.0],
 [15.0, -2.0, 0.0, 30.0, 3.0],
 [-5.0, -0.0, -2.0, 50.0, 1.0],
 [-0.0, -1.0, -0.0, 20.0, 3.0],
 [20.0, 1.0, 2.0, -0.0, 4.0],
 [20.0, 2.0, 4.0, -20.0, 1.0],
 [0.0, 0.0, 2.0, -10.0, -4.0],
 [0.0, 0.0, -2.0, 30.0, 0.0],
 [-15.0, 0.0, 4.0, 20.0, 4.0],
 [10.0, -2.0, 6.0, 30.0, -3.0],
 [-10.0, 2.0, 6.0, 40.0, -1.0],
 [0.0, 1.0, -2.0, 30.0, -2.0],
 [10.0, 0.0, 4.0, 20.0, 3.0],
 [-5.0, -0.0, -2.0, 10.0, 1.0],
 [15.0, -2.0, 2.0, -10.0, -2.0],
 [5.0, -0.0, 4.0, 30.0, -0.0],
 [-5.0, 2.0, 2.0, 30.

In [12]:
# Feasibility check: is the intersection of all halfspaces non-empty?
# We want to know if there exists theta such that for every i: dot(A[i], theta) > 0.
# If we find one theta, that's a proof the intersection is non-empty.

import math
import random
from typing import List, Tuple


def dot(u: List[float], v: List[float]) -> float:
    return sum(uu * vv for uu, vv in zip(u, v))


def l2norm(u: List[float]) -> float:
    return math.sqrt(sum(uu * uu for uu in u))


def perceptron_find_theta(
    A: List[List[float]],
    *,
    max_epochs: int = 20000,
    seed: int = 0,
) -> Tuple[bool, List[float], float]:
    """Try to find theta with dot(a_i, theta) > 0 for all i using a perceptron-style update.

    Returns: (found, theta, min_margin) where min_margin = min_i dot(a_i, theta).
    If found=False, theta is the last iterate and min_margin is the current minimum.
    """
    if not A:
        raise ValueError('A is empty')

    m = len(A)
    d = len(A[0])

    # Normalize constraints for stability (and detect any all-zero constraint)
    A_norm: List[List[float]] = []
    for i, a in enumerate(A):
        if len(a) != d:
            raise ValueError('Ragged A')
        n = l2norm(a)
        if n == 0:
            # 0 · theta > 0 is impossible
            return False, [0.0] * d, float('-inf')
        A_norm.append([ai / n for ai in a])

    theta = [0.0] * d

    rng = random.Random(seed)
    idxs = list(range(m))

    for _epoch in range(max_epochs):
        rng.shuffle(idxs)
        mistakes = 0
        for i in idxs:
            a = A_norm[i]
            if dot(a, theta) <= 0.0:
                # Update moves theta toward satisfying this constraint
                theta = [t + ai for t, ai in zip(theta, a)]
                mistakes += 1
        if mistakes == 0:
            min_margin = min(dot(a, theta) for a in A_norm)
            return True, theta, min_margin

    min_margin = min(dot(a, theta) for a in A_norm)
    return False, theta, min_margin


found, theta, min_margin = perceptron_find_theta(A)
print('heuristic (perceptron) found feasible theta?', found)
print('heuristic min dot(a_i, theta) over i:', min_margin)

if found:
    # You can rescale theta to get a cleaner margin if you want (e.g., min dot = 1)
    if min_margin > 0:
        scale = 1.0 / min_margin
        theta_scaled = [scale * t for t in theta]
        print('example feasible theta (scaled so min margin = 1):')
        print(theta_scaled)
else:
    print(
        "Heuristic did not find a feasible theta within the iteration budget. "
        "This alone does NOT prove infeasibility."
    )

# Certified check (recommended): solve a linear program for feasibility.
# Key fact: existence of theta with (A theta > 0) is equivalent to existence of theta with (A theta >= 1),
# because if A theta > 0 holds, you can scale theta to make all margins >= 1.

# We try to use scipy's LP solver if available.
try:
    import numpy as np
    from scipy.optimize import linprog

    if any(math.isnan(v) for row in A for v in row):
        raise ValueError('A contains NaNs; cannot certify feasibility until missing values are handled.')

    A_np = np.asarray(A, dtype=float)
    m, d = A_np.shape

    res = linprog(
        c=np.zeros(d),
        A_ub=-A_np,
        b_ub=-np.ones(m),
        bounds=[(None, None)] * d,
        method='highs',
    )

    print('\nCERTIFIED LP CHECK (A @ theta >= 1):')
    print('lp_success:', res.success)
    print('lp_status:', res.status)
    print('lp_message:', res.message)

    if res.success:
        margins = A_np @ res.x
        print('min_margin:', float(margins.min()))
        print('example feasible theta:', res.x)
    else:
        print('Conclusion: intersection of halfspaces is EMPTY for this dataset (no theta satisfies all constraints).')
except Exception as e:
    print('\nCertified LP check skipped (scipy/numpy not available in this environment):', repr(e))



heuristic (perceptron) found feasible theta? False
heuristic min dot(a_i, theta) over i: -0.6917412456440455
Heuristic did not find a feasible theta within the iteration budget. This alone does NOT prove infeasibility.

CERTIFIED LP CHECK (A @ theta >= 1):
lp_success: False
lp_status: 2
lp_message: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)
Conclusion: intersection of halfspaces is EMPTY for this dataset (no theta satisfies all constraints).
